# Le soft landing en une courbe : le chômage américain · *The soft landing in one curve: U.S. unemployment*

Notebook compagnon du chapitre **8. La carte du voyage : de la COVID au soft landing (2020-2025)** — [lire l'article](https://nmlab.io/ressources/de-la-covid-au-soft-landing-2020-2025).
Companion notebook to chapter **8. The Map of the Journey: From COVID to the Soft Landing (2020-2025)** — [read the article](https://nmlab.io/en/ressources/from-covid-to-the-soft-landing-2020-2025).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_data() -> tuple[Series, Series]:
    """Taux de chômage américain (UNRATE) et indicateur de récession NBER (USREC),
    2019-2025, en direct de FRED.
    U.S. unemployment rate and NBER recession flag, 2019-2025, live from FRED."""
    unrate = nm.load_fred("UNRATE", start="2019").loc["2019":"2025"].dropna()
    recessions = nm.load_fred("USREC", start="2019").loc["2019":"2025"]
    return unrate, recessions

unrate, recessions = load_data()


from pandas import Timestamp as T
import matplotlib.dates as mdates
from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="Le soft landing en une courbe : le chômage américain",
        band="récession NBER : 2 mois",
        peak="avr. 2020 : 14,8 % — record\nde la série d'après-guerre",
        start="févr. 2020 : 3,5 %",
        low="avr. 2023 : 3,4 %,\nplus bas depuis 1969",
        sahm="été 2024 : la règle de Sahm\ns'allume — fausse alerte",
        end="déc. 2025 : 4,4 %",
        note="Source : BLS via FRED (UNRATE). Mensuel, janv. 2019 – déc. 2025 — pas d'enquête ménages en oct. 2025.\n"
             "Bande rouge : récession NBER (févr.-avr. 2020), la plus courte jamais datée — et aucune depuis."),
    "en": dict(
        title="The soft landing in one curve: U.S. unemployment",
        band="NBER recession: 2 months",
        peak="Apr. 2020: 14.8% — record\nof the postwar series",
        start="Feb. 2020: 3.5%",
        low="Apr. 2023: 3.4%,\nlowest since 1969",
        sahm="Summer 2024: the Sahm rule\ntriggers — false alarm",
        end="Dec. 2025: 4.4%",
        note="Source: BLS via FRED (UNRATE). Monthly, Jan. 2019 – Dec. 2025 — no household survey in Oct. 2025.\n"
             "Red band: NBER recession (Feb.-Apr. 2020), the shortest ever dated — and none since."),
}

def build_figure(unrate: Series, recessions: Series, lang: str) -> Figure:
    """Chômage (aire remplie) + bande de récession NBER rose, cinq points annotés."""
    text = LABELS[lang]
    pct = FuncFormatter(lambda v, _: (f"{v:.0f} %" if lang == "fr" else f"{v:.0f}%"))
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig, left=0.06, top=0.86)

    runs = recessions.ne(recessions.shift()).cumsum()
    for _, run in recessions.groupby(runs):
        if run.iloc[0] == 1:
            ax.axvspan(run.index[0], run.index[-1], color=nm.COLORS["rose"], alpha=0.18, linewidth=0)
            mid = run.index[0] + (run.index[-1] - run.index[0]) / 2
            ax.text(mid, 10, text["band"], rotation=90, ha="center", va="center",
                    fontsize=17, color=nm.COLORS["rose"])

    ax.fill_between(unrate.index, unrate, color=nm.COLORS["blue"], alpha=0.13)
    ax.plot(unrate.index, unrate, color=nm.COLORS["blue"], linewidth=2.9)

    ax.set_ylim(0, 17)
    ax.set_yticks(range(0, 17, 2))
    ax.yaxis.set_major_formatter(pct)
    ax.set_xlim(T("2019-01-01"), T("2026-01-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    peak = unrate.loc["2020"].idxmax()
    low = unrate.loc["2022":"2024"].idxmin()
    ax.annotate(text["peak"], xy=(peak, unrate.loc[peak]), xytext=(T("2020-08-01"), 14.6),
                fontsize=21, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["start"], xy=(T("2020-02-01"), 3.5), xytext=(T("2019-01-20"), 5.7),
                fontsize=21, color=nm.COLORS["text"], va="center", ha="left",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["low"], xy=(low, unrate.loc[low]), xytext=(T("2021-11-01"), 7.1),
                fontsize=21, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["sahm"], xy=(T("2024-08-01"), unrate.loc["2024-08-01"]),
                xytext=(T("2023-06-01"), 10.6), fontsize=21, color=nm.COLORS["text"],
                va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["end"], xy=(unrate.index[-1], unrate.iloc[-1]), xytext=(T("2024-06-01"), 6),
                fontsize=21, color=nm.COLORS["text"], va="center", ha="left",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    nm.header(fig, text["title"])
    nm.footer(fig, text["note"])
    return fig

build_figure(unrate, recessions, LANG)